In [1]:
import duckdb as db
import pandas as pd
import time as time
import json
import requests
from kdtree import KdNode, KdTree
from boteco import Boteco
import csv
print("Ola")

Ola


In [2]:
con = db.connect(database=':memory:')

In [11]:
con.sql("""
        SELECT * FROM read_csv('butecos_bh.csv')
""")

┌─────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────┐
│            name             │                                          address                                          │
│           varchar           │                                          varchar                                          │
├─────────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────┤
│ 222                         │ R. Francisco Deslandes, 222 ,Belo Horizonte - MG, 30310-530                               │
│ Alexandre’s Bar             │ R. David Alves do Vale, 68  , Santa Rosa, Belo Horizonte - MG, 31255-630                  │
│ Andrade’s Beer              │ R. Dona Geni, 32 , Maria Helena, Belo Horizonte - MG, 31680-100                           │
│ Armazém Santa Amélia        │ Rua da Sinfonia, 253 , Santa Amélia, Belo Horizonte - MG, 31560-420                       │
│ Avalan

In [12]:
con.sql("""
        select * from read_csv('butecos_bh.csv') where name like 'Já Tô Inno'            
""")

┌────────────┬──────────────────────────────────────────────────────────────────┐
│    name    │                             address                              │
│  varchar   │                             varchar                              │
├────────────┼──────────────────────────────────────────────────────────────────┤
│ Já Tô Inno │ R. Benjamim Dias, 379 , Barreiro, Belo Horizonte - MG, 30640-520 │
└────────────┴──────────────────────────────────────────────────────────────────┘

In [ ]:
#https://www.youtube.com/watch?v=H6hkSADi1Rc
#https://www.youtube.com/watch?v=rmIhGPy8rSY
url = "https://nominatim.openstreetmap.org/search"
parametros = {
    "q": "R. Benjamim Dias, Barreiro, Belo Horizonte",
    "format": "json"
}
cabecalhos = {
    "User-Agent": "davi" 
}

retorno = requests.get(url, params=parametros, headers=cabecalhos)
print(retorno.json())

[]


In [ ]:
botecos = []
url = "https://nominatim.openstreetmap.org/search"

cabecalhos = {
    "User-Agent": "projeto_tp1_algoritmos_botecos" 
}

resultados = con.execute("""
        SELECT * FROM read_csv('butecos_bh.csv')
""").fetchall()
for tupla in resultados:
    parametros = {"q": f"{tupla[1]}", "format": "json"}
    retorno = requests.get(url, params=parametros, headers=cabecalhos)
    if retorno.status_code == 200:
        retorno = retorno.json()
        if len(retorno) >=1:
            botecos.append(Boteco(tupla[0], float(retorno[0]['lon']), float(retorno[0]['lat']), tupla[1]))
    else: 
        print(f"Erro HTTP {retorno.status_code} no bar: {tupla[0]}")
    time.sleep(1)

222
Alexandre’s Bar
Andrade’s Beer
Avalanche Bar e Restaurante
Azougue Fogo e Bar
Baiúca
Bar Bambú
Bar Bendita Baderna
Bar da Cintia
Bar da Fia
Bar da Gisa
Bar da Lu
Bar da Praça
,Bar da Silvânia
Bar do Bartolomeu
Bar do Bem
Bar do Dedinho
Bar do João (São João Batista)
Bar do Momô
Bar do Nelson
Bar do Peixinho
Bar do Primo
Bar do Regis
Bar do Tião
Bar dos Meninos
Bar Du Magrelo
Bar e Restaurante Bom Sabor
Bar e Restaurante do Branco
Bar Estabelecimento
Bar Junto Juntinho
Bar Mania Mineira
Bar Pompéu
Bar Stella
Barrigudinha Buteco
Barzim dos Amigos
Beco Restaurante
Bella Bar e Restaurante
Boi Major
Boteco 86
Botequim Buritis
Butiquim 325
Butiquim On Cê Tá?
Café Palhares
Camisola Bar
Cantina Arte Quintal
Cantinho da Baiana
Casa Mojubá
Chapa Mágica
Choperia América Norte Sul
Chopp da Esquina
Cipoeiro Bar
Comida & Prosa
Companhia do Dino
Conectados Bar
COSMOS
Dona Dora
Dona Ju Gastro Bar
Dona Suica
Empório Laís
Espetinho do Boi
Espetinho do Tilias
Espetinho Rei e Família
Espetinho São Ped

In [18]:
with open('botecos_coordenadas.csv', mode='w', newline='', encoding='utf-8') as csvFile:
    
   
    csvWriter = csv.writer(csvFile, delimiter=';')
    
   
    csvWriter.writerow(['nome', 'lon', 'lat', 'endereco'])
    
   
    for botecoObj in botecos:
        csvWriter.writerow([
            botecoObj.nome, 
            botecoObj.lon, 
            botecoObj.lat, 
            botecoObj.endereco
        ])

In [3]:
botecos = []
resultados = con.execute("""
        SELECT * FROM read_csv('botecos_coordenadas.csv')
""").fetchall()
for quad in resultados:
    botecos.append(Boteco(quad[0], quad[1], quad[2], quad[3]))

In [6]:
arvore = KdTree()
arvore.insertAll(Botecos=botecos)
botecosNaArea = []
minx = -43.9500
maxx = -43.9300
miny = -19.9300
maxy = -19.9100
arvore.search(minx, maxx, miny, maxy, botecosNaArea, 0, arvore.raiz)
print(f"Total de bares encontrados no retângulo: {len(botecosNaArea)}")
print("-" * 40)
for boteco in botecosNaArea:
    print(f"{boteco.nome} (Lon: {boteco.lon:.4f}, Lat: {boteco.lat:.4f})")

Total de bares encontrados no retângulo: 4
----------------------------------------
Dona Dora (Lon: -43.9405, Lat: -19.9258)
Beco Restaurante (Lon: -43.9455, Lat: -19.9188)
Graffica Bar (Lon: -43.9376, Lat: -19.9248)
Café Palhares (Lon: -43.9443, Lat: -19.9164)
